In [1]:
from transport.routes_generator import citygraph_dataset
from transport.routes_generator import inductive_route_learning, eval_route_generator, bee_colony
from transport.routes_generator.bee_colony import main as main_bee  # я так обозвал
from transport.routes_generator.eval_route_generator import main as main_eval # я так обозвал
from omegaconf import OmegaConf, DictConfig

from tqdm import tqdm
from pathlib import Path

# Adaptation function

In [ ]:
from hydra import compose, initialize_config_dir

def get_eval_cfg(cfg_dir: str, base_cfg_name: str = "eval_model_mumford", params: dict | None = None):
    """
    Creates a Hydra config for model evaluation.

    params — dictionary with "human-friendly" parameter names, e.g.:
        {
            "dataset_name": "mumford",
            "n_routes": 12,
            "min_route_len": 2,
            "max_route_len": 15,
            "demand_time_weight": 0.3,
            "route_time_weight": 0.4,
            "median_connectivity_weight": 0.3,
            "run_name": "my_test_run",
            "model_weights": "../path/to/model.pt"
        }

    The function automatically maps the keys to Hydra config paths.
    Unknown keys are ignored.
    """
    if params is None:
        params = {}

    # mapping "human-friendly" keys to Hydra paths
    key_map = {
        "dataset_name": "+eval.dataset.type",  # добавляем новый ключ
        "n_routes": "+eval.n_routes",          # добавляем новый ключ, если нет
        "min_route_len": "+eval.min_route_len",
        "max_route_len": "+eval.max_route_len",
        "demand_time_weight": "++experiment.cost_function.kwargs.demand_time_weight",
        "route_time_weight": "++experiment.cost_function.kwargs.route_time_weight",
        "median_connectivity_weight": "++experiment.cost_function.kwargs.median_connectivity_weight",
        "constraint_violation_weight": "++experiment.cost_function.kwargs.constraint_violation_weight",
        "use_weighted_connectivity": "++experiment.cost_function.kwargs.use_weighted_connectivity",
        "variable_weights": "++experiment.cost_function.kwargs.variable_weights",
        "pp_fraction": "++experiment.cost_function.kwargs.pp_fraction",
        "op_fraction": "++experiment.cost_function.kwargs.op_fraction",
        "mcw_fraction": "++experiment.cost_function.kwargs.mcw_fraction",
        "run_name": "++run_name",
        "model_weights": "++model.weights",
        "starting_routes_init": "++init.method",
        "starting_routes_path":"++init.path"
    }

    # формируем список overrides
    overrides = []
    for k, v in params.items():
        path = key_map.get(k)
        if path:
            overrides.append(f"{path}={v}")
        else:
            print(f"[Warning] Ignored unknown parameter: {k}")

    with initialize_config_dir(config_dir=cfg_dir, version_base=None):
        cfg_eval = compose(config_name=base_cfg_name, overrides=overrides)

    return cfg_eval


# Read data

## Read Mandl

In [ ]:
# import torch
# import numpy as np
# from pathlib import Path

# # Пути к файлам
# instances_dir = r"/root/transport/data"
# instance_name = "Mandl"

# data_dir = Path(instances_dir)

# # Чтение координат
# coords_path = data_dir / (instance_name + 'Coords.txt')
# node_locs = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)
# orig_pos = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)

# # Чтение времени путешествия
# tt_path = data_dir / (instance_name + 'TravelTimes.txt')
# street_adj = torch.tensor(np.genfromtxt(tt_path), dtype=torch.float32) * 60

# # Чтение спроса
# dmd_path = data_dir / (instance_name + 'Demand.txt')
# demand = torch.tensor(np.genfromtxt(dmd_path), dtype=torch.float32)

# # Результаты
# print("Coords tensor shape:", node_locs.shape)
# print("Travel times tensor shape:", street_adj.shape)
# print("Demand tensor shape:", demand.shape)

# input_tensors = {            
#     'street_adj':street_adj,
#     'demand':demand,    
#     'node_locs':node_locs
#     }

## Read Tartu

In [22]:
import pickle
import pandas as pd
import numpy as np
import torch
import networkx as nx

# --- 1. Загрузка графа G ---
file_path = '/root/transport/examples/data/bus_graph_Tartu.pkl'
with open(file_path, 'rb') as f:
    G = pickle.load(f)

# --- 2. Загрузка OD-матрицы ---
OD = pd.read_csv('/root/transport/examples/data/OD_matrix_Tartu.csv').set_index('Unnamed: 0', drop=True)
demand_np = OD.values  # numpy array
demand = torch.tensor(demand_np, dtype=torch.float32)

# --- 3. Получение списка узлов в правильном порядке ---
nodes = sorted(G.nodes())  # Важно: сортировка, чтобы индексы совпадали с OD
n_nodes = len(nodes)
node_to_idx = {node: i for i, node in enumerate(nodes)}

# --- 4. Матрица координат (node_locs): [x, y] ---
node_locs_list = []
for node in nodes:
    x = G.nodes[node]['x']
    y = G.nodes[node]['y']
    node_locs_list.append([x, y])

node_locs_np = np.array(node_locs_list, dtype=np.float32)
node_locs = torch.tensor(node_locs_np)

# --- 5. Симметричная матрица весов по time_min (street_adj) ---
inf = float('inf')
adj_matrix = np.full((n_nodes, n_nodes), inf, dtype=np.float32)
np.fill_diagonal(adj_matrix, 0.0)  # диагональ = 0

for u, v, data in G.edges(data=True):
    if 'time_min' in data:
        time = data['time_min']
        i = node_to_idx[u]
        j = node_to_idx[v]
        adj_matrix[i, j] = time
        adj_matrix[j, i] = time  # симметричность

street_adj = torch.tensor(adj_matrix)
# Замена np.inf → torch.inf
street_adj[street_adj == inf] = torch.inf

# --- 6. Вывод результата ---
input_tensors = {
    'street_adj': street_adj,
    'demand': demand,
    'node_locs': node_locs
}

print("Готово! Формат:")
print(f"street_adj: {street_adj.shape} → {street_adj.dtype}")
print(f"demand: {demand.shape} → {demand.dtype}")
print(f"node_locs: {node_locs.shape} → {node_locs.dtype}")

# Пример первых строк
print("\nstreet_adj[0:5, 0:5]:\n", street_adj[:5, :5])
print("\nnode_locs[:5]:\n", node_locs[:5])
print("\ndemand[0:5, 0:5]:\n", demand[:5, :5])

Готово! Формат:
street_adj: torch.Size([199, 199]) → torch.float32
demand: torch.Size([199, 199]) → torch.float32
node_locs: torch.Size([199, 2]) → torch.float32

street_adj[0:5, 0:5]:
 tensor([[0.0000, 2.2362,    inf,    inf,    inf],
        [2.2362, 0.0000,    inf,    inf,    inf],
        [   inf,    inf, 0.0000,    inf,    inf],
        [   inf,    inf,    inf, 0.0000,    inf],
        [   inf,    inf,    inf,    inf, 0.0000]])

node_locs[:5]:
 tensor([[ 482896.5938, 6468276.0000],
        [ 482234.9062, 6468179.0000],
        [ 482957.5312, 6473988.5000],
        [ 483041.6875, 6473111.5000],
        [ 487566.2812, 6467616.5000]])

demand[0:5, 0:5]:
 tensor([[ 0.0000,  8.7426,  0.1264,  0.6597,  0.9253],
        [ 8.0729,  0.0000,  1.7791,  9.4088, 14.3724],
        [ 0.1146,  1.7471,  0.0000,  2.2898,  0.8077],
        [ 0.2799,  4.3225,  1.0712,  0.0000,  2.0049],
        [ 0.6897, 11.6007,  0.6639,  3.5225,  0.0000]])


# Run LC

In [ ]:
from omegaconf import OmegaConf
import os

# params полностью повторяет значения из твоего примера
params = {
    "dataset_name": "tensor",
    "n_routes": 6,
    "min_route_len": 2,
    "max_route_len": 8,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "test_run",
    "model_weights": "../transport/routes_generator/inductive_random_graphs_weighted_connectivity.pt"
}

cfg_dir = os.path.abspath("../transport/routes_generator/cfg")

cfg_eval = get_eval_cfg(
    cfg_dir=cfg_dir,
    base_cfg_name='eval_model_mumford',
    params=params
)

# print(OmegaConf.to_yaml(cfg_eval))


In [24]:
from torch_geometric.loader import DataLoader
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
import transport.routes_generator.utils as lrnu
from transport.routes_generator.eval_route_generator import eval_model
from transport.routes_generator.torch_utils import dump_routes

cfg = cfg_eval

# Если у тебя есть заранее подготовленные тензоры — можешь загрузить здесь:
tensors = input_tensors

# Стандартная обработка конфига и модели
assert 'model' in cfg, "Must provide config for model!"
DEVICE, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
    cfg, 'nn_construction_', weights_required=True
)

# Загружаем тестовый датасет
test_ds = get_dataset_from_config(cfg.eval.dataset, tensors=tensors)
test_dl = DataLoader(test_ds, batch_size=cfg.batch_size)

# Запускаем оценку модели
n_samples = cfg.get('n_samples', None)
sbs = cfg.get('sample_batch_size', cfg.batch_size)


_, unserved_demand, metrics, routes = eval_model(
    model,
    test_dl,
    cfg.eval,
    cost_obj,
    n_samples=n_samples,
    sample_batch_size=sbs,
    return_routes=True,
    silent=True,
    device=DEVICE
)

# Сохраняем маршруты
dump_routes(run_name, routes.cpu())

# Печатаем результат в консоль
print("=== Evaluation done ===")
print(f"Run name: {run_name}")
print(f"Unserved demand: {unserved_demand}")
print(f"Metrics: {metrics}")
print(routes)

/root/transport/transport/routes_generator/utils.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg.model.weights,


=== Evaluation done ===
Run name: nn_construction_test_run
Unserved demand: tensor([[[0.0000e+00, 8.7426e+00, 1.2635e-01,  ..., 5.6898e+00,
          1.1788e-01, 2.7935e+00],
         [8.0729e+00, 0.0000e+00, 1.7791e+00,  ..., 1.3244e+02,
          1.6295e+00, 6.0707e+01],
         [1.1458e-01, 1.7471e+00, 0.0000e+00,  ..., 1.9662e+00,
          3.8255e-01, 1.0190e+00],
         ...,
         [2.1393e+00, 5.3928e+01, 8.1524e-01,  ..., 0.0000e+00,
          7.4514e-01, 6.5929e+01],
         [2.3921e-01, 3.5809e+00, 8.5605e-01,  ..., 4.0217e+00,
          0.0000e+00, 2.1025e+00],
         [7.7934e+00, 1.8341e+02, 3.1349e+00,  ..., 4.8918e+02,
          2.8904e+00, 0.0000e+00]]])
Metrics: {'cost': tensor([6.3247]), 'ATT': tensor([1.5574]), 'RTT': tensor([2.0098]), '$d_0$': tensor([1.9226]), '$d_1$': tensor([0.8100]), '$d_2$': tensor([0.]), '$d_{un}$': tensor([97.2675]), '# disconnected node pairs': tensor([18363.5000]), '# stops out of bounds': tensor([0.]), 'median_connectivity': tensor(

/root/transport/transport/routes_generator/utils.py:317: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


# Run NBCO 

In [25]:
from omegaconf import OmegaConf
import os

# params полностью повторяет значения из твоего примера
params = {
    "dataset_name": "tensor",
    "n_routes": 6,
    "min_route_len": 2,
    "max_route_len": 8,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "test_run",
    "model_weights": "../transport/routes_generator/inductive_random_graphs_weighted_connectivity.pt",
    "starting_routes_init":"tensor"
}

cfg_dir = os.path.abspath("../transport/routes_generator/cfg")

cfg_bee = get_eval_cfg(
    cfg_dir=cfg_dir,
    base_cfg_name='neural_bco_mumford',
    params=params
)

# print(OmegaConf.to_yaml(cfg_bee))


In [ ]:
from torch_geometric.loader import DataLoader
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
import transport.routes_generator.utils as lrnu

use_neural_bees = cfg_bee.get('neural_bees', False)
if use_neural_bees:
    prefix = 'neural_bco_'
else:
    prefix = 'bco_'

DEVICE, run_name, sum_writer, cost_obj, bee_model = \
    lrnu.process_standard_experiment_cfg(cfg_bee, prefix, 
                                            weights_required=True)

# read in the dataset
test_ds = get_dataset_from_config(cfg_bee.eval.dataset, tensors=input_tensors)
test_dl = DataLoader(test_ds, batch_size=cfg_bee.batch_size)

force_linking_unlinked = cfg_bee.get('force_linking_unlinked', False)

if not use_neural_bees:
    bee_model = None
elif bee_model is not None:
    bee_model.force_linking_unlinked = force_linking_unlinked
    bee_model.eval()

nt1b = cfg_bee.get('n_type1_bees', None)
nt2b = cfg_bee.get('n_type2_bees', None)
test_output = \
    lrnu.test_method(bee_colony, test_dl, cfg_bee.eval, cfg_bee.init, cost_obj, 
        sum_writer=sum_writer, silent=True, n_bees=cfg_bee.n_bees,
        n_iterations=cfg_bee.n_iterations, n_type1_bees=nt1b, n_type2_bees=nt2b,  
        device=DEVICE, bee_model=bee_model, return_routes=True,
        force_linking_unlinked=force_linking_unlinked, routes_tensor=routes)
routes = test_output[-1]
metrics = test_output[-2]
unserved_demand = test_output[-3]

# Печатаем результат в консоль
print("=== Evaluation done ===")
print(f"Run name: {run_name}")
print(f"Unserved demand: {unserved_demand}")
print(f"Metrics: {metrics}")
print(routes)